In [1]:
# ===========================================================
# Section 1 — Setup: Install required libraries
# ===========================================================
# Tip: Run this cell first and restart runtime if prompted.

import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("transformers", "torch", "sentence-transformers")
pip_install("faiss-cpu")          # CPU-only FAISS — works on free Colab
pip_install("PyPDF2", "pdfplumber")
pip_install("gradio")             # Optional web UI

print("✅ All libraries installed.")


✅ All libraries installed.


In [2]:
# ===========================================================
# Section 2 — Imports
# ===========================================================

import os, time, json, pickle, textwrap
import numpy as np

# HuggingFace QA pipeline
from transformers import pipeline

# Sentence-Transformers for dense embeddings
from sentence_transformers import SentenceTransformer

# FAISS (optional — falls back to numpy cosine similarity)
try:
    import faiss
    FAISS_AVAILABLE = True
    print("✅ FAISS available — using FAISS index.")
except ImportError:
    FAISS_AVAILABLE = False
    print("⚠️  FAISS not found — falling back to numpy cosine similarity.")

# PDF parsing
try:
    import pdfplumber
    PDFPLUMBER_AVAILABLE = True
except ImportError:
    PDFPLUMBER_AVAILABLE = False

try:
    import PyPDF2
    PYPDF2_AVAILABLE = True
except ImportError:
    PYPDF2_AVAILABLE = False

# Colab file upload helper
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("ℹ️  Not running in Colab — file upload widget disabled.")

# ===========================================================
# Section 3 — Hyperparameters / Configuration
# ===========================================================
# Adjust these to trade speed for accuracy.

CHUNK_SIZE   = 500    # characters per chunk (~400 tokens for English)
CHUNK_STRIDE = 100    # overlap between consecutive chunks
TOP_K        = 3      # number of chunks to retrieve per question
MAX_ANS_LEN  = 100    # max answer length (tokens) for extractive QA
MEMORY_TURNS = 3      # how many past Q/A pairs to prepend to next query

QA_MODEL_NAME  = "deepset/roberta-base-squad2"
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

INDEX_PATH      = "qa_faiss.index"      # saved FAISS index
EMBEDDINGS_PATH = "qa_embeddings.pkl"   # saved embeddings + chunks

print(f"Config — chunk_size={CHUNK_SIZE}, stride={CHUNK_STRIDE}, "
      f"top_k={TOP_K}, memory_turns={MEMORY_TURNS}")


✅ FAISS available — using FAISS index.
Config — chunk_size=500, stride=100, top_k=3, memory_turns=3


In [4]:
# ===========================================================
# Section 4 — Load Models
# ===========================================================

print("\n⏳ Loading QA model (first run downloads ~500 MB)…")
t0 = time.time()
try:
    qa_pipeline = pipeline(
        "question-answering",
        model=QA_MODEL_NAME,
        tokenizer=QA_MODEL_NAME,
        max_answer_len=MAX_ANS_LEN,
    )
    print(f"✅ QA model loaded in {time.time()-t0:.1f}s")
except Exception as e:
    print(f"❌ QA model failed to load: {e}")
    raise

print("\n⏳ Loading embedding model (~90 MB)…")
t0 = time.time()
try:
    emb_model = SentenceTransformer(EMB_MODEL_NAME)
    print(f"✅ Embedding model loaded in {time.time()-t0:.1f}s")
except Exception as e:
    print(f"❌ Embedding model failed to load: {e}")
    raise

# ===========================================================
# Section 5 — Upload Documents
# ===========================================================
# Supported: .txt and .pdf files.
# In Colab, a file-picker dialog will appear.
# Outside Colab, place files in the working directory manually
# and set DEMO_MODE = True to auto-generate sample docs.

DEMO_MODE = not IN_COLAB   # set True to skip upload and use sample text

def create_sample_documents():
    """Generate sample .txt files so the demo works without uploads."""
    docs = {
        "apollo11.txt": (
            "The Apollo 11 mission was the first crewed lunar landing mission "
            "in history. It launched on July 16, 1969, from Kennedy Space Center "
            "in Florida. The mission was led by Commander Neil Armstrong, with "
            "Command Module Pilot Michael Collins and Lunar Module Pilot Buzz Aldrin. "
            "On July 20, 1969, Armstrong and Aldrin landed on the Moon in the Lunar "
            "Module Eagle while Collins orbited above in Columbia. Armstrong became "
            "the first human to step onto the lunar surface, followed by Aldrin. "
            "The main mission objective was to land humans on the Moon and return them "
            "safely to Earth, fulfilling President Kennedy's 1961 national goal. "
            "The crew splashed down in the Pacific Ocean on July 24, 1969, completing "
            "an eight-day journey. The mission collected 47.5 pounds of lunar material "
            "and deployed several scientific instruments on the surface."
        ),
        "climate_change.txt": (
            "Climate change refers to long-term shifts in global temperatures and "
            "weather patterns. While natural factors have historically driven climate "
            "change, since the mid-20th century human activities — primarily burning "
            "fossil fuels — have been the main driver. The Intergovernmental Panel on "
            "Climate Change (IPCC) was established in 1988 to assess the science of "
            "climate change. Its reports synthesise thousands of scientific papers to "
            "provide policymakers with regular assessments. The Paris Agreement, adopted "
            "in December 2015, brought nearly every nation together to combat climate "
            "change and limit global warming to well below 2°C above pre-industrial "
            "levels. Key effects include rising sea levels, more frequent extreme weather "
            "events, loss of biodiversity, and disruptions to food systems. Renewable "
            "energy adoption, reforestation, and carbon capture are among the principal "
            "mitigation strategies recommended by scientists worldwide."
        ),
        "deep_learning.txt": (
            "Deep learning is a subset of machine learning that uses neural networks "
            "with many layers to learn representations from raw data. The field was "
            "significantly advanced by Geoffrey Hinton, Yann LeCun, and Yoshua Bengio, "
            "who were awarded the Turing Award in 2018 for their foundational "
            "contributions. The ImageNet competition, held annually from 2010 onward, "
            "became a milestone when AlexNet — developed by Krizhevsky, Sutskever, and "
            "Hinton — dramatically outperformed traditional methods in 2012. Transformer "
            "architectures, introduced in the paper 'Attention Is All You Need' by "
            "Vaswani et al. in 2017, revolutionised natural language processing and "
            "later computer vision. Large language models such as GPT, BERT, and their "
            "successors are trained on vast corpora and can perform a wide range of "
            "tasks with little or no task-specific fine-tuning. Deep learning now powers "
            "speech recognition, machine translation, recommendation systems, drug "
            "discovery, and autonomous vehicles."
        ),
    }
    for fname, content in docs.items():
        with open(fname, "w") as f:
            f.write(content)
    print(f"✅ Created {len(docs)} sample documents: {list(docs.keys())}")
    return list(docs.keys())

# ── Text extraction helpers ──────────────────────────────────

def extract_text_from_pdf(path: str) -> str:
    """Try pdfplumber first, fall back to PyPDF2."""
    text = ""
    if PDFPLUMBER_AVAILABLE:
        try:
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
            return text
        except Exception as e:
            print(f"  pdfplumber failed ({e}), trying PyPDF2…")

    if PYPDF2_AVAILABLE:
        try:
            with open(path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                for page in reader.pages:
                    text += page.extract_text() or ""
            return text
        except Exception as e:
            print(f"  PyPDF2 failed ({e})")

    print(f"  ⚠️  Could not extract text from {path} — no PDF library available.")
    return ""

def load_documents(file_paths: list) -> dict:
    """Return {filename: raw_text} for all uploaded files."""
    docs = {}
    for path in file_paths:
        name = os.path.basename(path)
        ext  = name.rsplit(".", 1)[-1].lower()
        try:
            if ext == "txt":
                with open(path, "r", errors="ignore") as f:
                    docs[name] = f.read()
            elif ext == "pdf":
                docs[name] = extract_text_from_pdf(path)
            else:
                print(f"  ⚠️  Unsupported file type: {name} — skipping.")
                continue
            print(f"  📄 {name}: {len(docs[name])} characters")
        except Exception as e:
            print(f"  ❌ Could not read {name}: {e}")
    return docs

# ── Upload or use demo files ─────────────────────────────────

if DEMO_MODE:
    print("📝 Demo mode — generating sample documents…")
    uploaded_paths = create_sample_documents()
else:
    print("📂 Please select .txt or .pdf files using the upload widget…")
    uploaded = colab_files.upload()   # shows Colab file-picker
    uploaded_paths = list(uploaded.keys())
    print(f"✅ Uploaded: {uploaded_paths}")

raw_documents = load_documents(uploaded_paths)

if not raw_documents:
    raise RuntimeError("No documents loaded — cannot continue.")

# ===========================================================
# Section 6 — Preprocess: Chunking
# ===========================================================
# Split each document into overlapping character-level chunks.
# Overlap (stride) helps the QA model see cross-boundary context.

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               stride: int = CHUNK_STRIDE) -> list[str]:
    """
    Sliding-window chunker.
    chunk_size : target chunk length in characters
    stride     : step size; overlap = chunk_size - stride
    """
    chunks = []
    start  = 0
    text   = text.strip()
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += stride   # slide forward by stride, keeping overlap
    return chunks

# Build a flat list of (source_doc, chunk_text) tuples
all_chunks   = []   # list of chunk strings
chunk_sources = []  # parallel list of source filenames

for doc_name, doc_text in raw_documents.items():
    doc_chunks = chunk_text(doc_text)
    all_chunks.extend(doc_chunks)
    chunk_sources.extend([doc_name] * len(doc_chunks))
    print(f"  📑 {doc_name} → {len(doc_chunks)} chunks")

print(f"\n✅ Total chunks: {len(all_chunks)}")


⏳ Loading QA model (first run downloads ~500 MB)…


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ QA model loaded in 5.1s

⏳ Loading embedding model (~90 MB)…


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded in 4.6s
📂 Please select .txt or .pdf files using the upload widget…


Saving 2026008103 (1).pdf to 2026008103 (1).pdf
✅ Uploaded: ['2026008103 (1).pdf']
  📄 2026008103 (1).pdf: 31028 characters
  📑 2026008103 (1).pdf → 307 chunks

✅ Total chunks: 307


In [5]:
# ===========================================================
# Section 7 — Build Index
# ===========================================================
# Encode every chunk into a dense vector, then index with FAISS.
# Falls back to plain numpy cosine search if FAISS is unavailable.

def build_embeddings(chunks: list[str]) -> np.ndarray:
    """Encode chunks with the SentenceTransformer; returns float32 array."""
    print(f"⏳ Encoding {len(chunks)} chunks…")
    t0 = time.time()
    embeddings = emb_model.encode(
        chunks,
        show_progress_bar=True,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,   # L2-normalise → cosine via dot product
    )
    print(f"✅ Encoded in {time.time()-t0:.1f}s | shape: {embeddings.shape}")
    return embeddings.astype("float32")

def build_faiss_index(embeddings: np.ndarray):
    """Build an exact inner-product FAISS index (cosine on normalised vecs)."""
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)   # Inner Product (= cosine for unit vecs)
    index.add(embeddings)
    print(f"✅ FAISS index built — {index.ntotal} vectors, dim={dim}")
    return index

# Check if saved index exists; load or rebuild
if os.path.exists(INDEX_PATH) and os.path.exists(EMBEDDINGS_PATH):
    print("📂 Found saved index — loading from disk…")
    with open(EMBEDDINGS_PATH, "rb") as f:
        saved = pickle.load(f)
    chunk_embeddings = saved["embeddings"]
    # Note: chunks/sources loaded from disk override the ones built above
    all_chunks    = saved["chunks"]
    chunk_sources = saved["sources"]
    if FAISS_AVAILABLE:
        faiss_index = faiss.read_index(INDEX_PATH)
        print(f"✅ FAISS index loaded — {faiss_index.ntotal} vectors")
    else:
        faiss_index = None
else:
    chunk_embeddings = build_embeddings(all_chunks)

    if FAISS_AVAILABLE:
        faiss_index = build_faiss_index(chunk_embeddings)
        faiss.write_index(faiss_index, INDEX_PATH)
        print(f"💾 FAISS index saved → {INDEX_PATH}")
    else:
        faiss_index = None
        print("ℹ️  Skipping FAISS save (not available).")

    # Save embeddings + chunk metadata for future reuse
    with open(EMBEDDINGS_PATH, "wb") as f:
        pickle.dump({
            "embeddings": chunk_embeddings,
            "chunks":     all_chunks,
            "sources":    chunk_sources,
        }, f)
    print(f"💾 Embeddings saved → {EMBEDDINGS_PATH}")

# ===========================================================
# Section 8 — Retrieval
# ===========================================================

def cosine_similarity_numpy(query_vec: np.ndarray,
                             corpus_vecs: np.ndarray) -> np.ndarray:
    """Fallback: brute-force cosine similarity (vecs already L2-normalised)."""
    return corpus_vecs @ query_vec   # dot product = cosine for unit vectors

def retrieve_chunks(question: str, top_k: int = TOP_K) -> list[dict]:
    """
    Given a natural-language question, encode it, search the index,
    and return top_k chunks as dicts with keys: text, source, score.
    """
    # Encode and normalise the query vector
    q_vec = emb_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    if FAISS_AVAILABLE and faiss_index is not None:
        scores, indices = faiss_index.search(q_vec, top_k)
        scores, indices = scores[0], indices[0]
    else:
        sims    = cosine_similarity_numpy(q_vec[0], chunk_embeddings)
        indices = np.argsort(sims)[::-1][:top_k]
        scores  = sims[indices]

    results = []
    for idx, score in zip(indices, scores):
        if idx < 0:   # FAISS returns -1 for unfilled slots
            continue
        results.append({
            "text":   all_chunks[idx],
            "source": chunk_sources[idx],
            "score":  float(score),
        })
    return results

def assemble_context(chunks: list[dict]) -> str:
    """Concatenate retrieved chunks into a single context string for QA."""
    parts = []
    for i, c in enumerate(chunks, 1):
        parts.append(f"[Chunk {i} — {c['source']}]\n{c['text']}")
    return "\n\n".join(parts)


⏳ Encoding 307 chunks…


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Encoded in 20.4s | shape: (307, 384)
✅ FAISS index built — 307 vectors, dim=384
💾 FAISS index saved → qa_faiss.index
💾 Embeddings saved → qa_embeddings.pkl


In [6]:
# ===========================================================
# Section 9 — Question Answering
# ===========================================================

def answer_question(question: str,
                    context: str,
                    top_k_answers: int = 1) -> dict:
    """
    Run extractive QA over the assembled context.
    Returns the best answer dict: {answer, score, start, end}.
    """
    if not context.strip():
        return {"answer": "No context available.", "score": 0.0}
    try:
        result = qa_pipeline(
            question=question,
            context=context,
            top_k=top_k_answers,
        )
        # pipeline returns a list when top_k>1, else a single dict
        if isinstance(result, list):
            return result[0]
        return result
    except Exception as e:
        return {"answer": f"QA error: {e}", "score": 0.0}

# ===========================================================
# Section 10 — Conversational Memory
# ===========================================================
# A minimal rolling buffer of the last MEMORY_TURNS Q/A pairs.
# Prepending them to the question gives the model useful context
# for follow-up questions.

conversation_history: list[dict] = []   # each: {question, answer}

def build_query_with_memory(new_question: str) -> str:
    """
    Prepend recent Q/A history to the new question so the retriever
    and QA model can leverage conversational context.
    """
    if not conversation_history:
        return new_question
    # Use only the last MEMORY_TURNS turns
    recent = conversation_history[-MEMORY_TURNS:]
    memory_str = " ".join(
        f"Q: {t['question']} A: {t['answer']}" for t in recent
    )
    return f"{memory_str} Follow-up: {new_question}"

def chat(question: str, top_k: int = TOP_K, verbose: bool = True) -> str:
    """
    Full pipeline:
      1. Expand question with memory
      2. Retrieve relevant chunks
      3. Assemble context
      4. Run extractive QA
      5. Store turn in memory
      6. Return answer string
    """
    # Step 1 — augment with memory
    augmented_q = build_query_with_memory(question)

    # Step 2 — retrieve
    chunks = retrieve_chunks(augmented_q, top_k=top_k)

    if verbose:
        print(f"\n🔍 Retrieved {len(chunks)} chunk(s):")
        for c in chunks:
            print(f"   [{c['source']}] score={c['score']:.3f} — "
                  f"{c['text'][:80].replace(chr(10),' ')}…")

    # Step 3 — assemble context
    context = assemble_context(chunks)

    # Step 4 — QA
    result  = answer_question(question, context)
    answer  = result.get("answer", "")
    score   = result.get("score", 0.0)

    # Step 5 — store in memory
    conversation_history.append({"question": question, "answer": answer})

    if verbose:
        print(f"\n💬 Q: {question}")
        print(f"✅ A: {answer}  (confidence={score:.3f})")

    return answer

In [7]:
# ===========================================================
# Section 11 — Demo QA Interactions
# ===========================================================
# Three example interactions showcasing single-chunk, multi-chunk,
# and conversational follow-up retrieval.

print("\n" + "="*60)
print("DEMO — 3 Sample QA Interactions")
print("="*60)

# Reset memory for a clean demo
conversation_history.clear()

# ── Demo 1: Factual question answerable from one chunk ───────
print("\n--- Demo 1: Single-chunk factual question ---")
chat("What is the main mission described in the uploaded document?")

# ── Demo 2: Question spanning multiple document chunks ───────
print("\n--- Demo 2: Multi-chunk question ---")
chat("When did the event described in the text happen?")

# ── Demo 3: Follow-up using conversational memory ────────────
print("\n--- Demo 3: Follow-up using memory ---")
chat("And who led that effort?")

# ===========================================================
# Section 12 — Interactive Chat Loop (Terminal / Colab)
# ===========================================================
# Type your question and press Enter. Type 'quit' to exit.
# Memory is preserved across turns within the session.

print("\n" + "="*60)
print("Interactive Chat Loop")
print("Type your question and press Enter. Type 'quit' to exit.")
print("="*60)

def interactive_loop():
    conversation_history.clear()   # fresh memory for interactive session
    while True:
        try:
            user_input = input("\n❓ Your question: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Exiting chat loop.")
            break

        if not user_input:
            continue
        if user_input.lower() in {"quit", "exit", "q"}:
            print("👋 Goodbye!")
            break
        if user_input.lower() == "clear":
            conversation_history.clear()
            print("🗑️  Conversation memory cleared.")
            continue

        chat(user_input)

# Uncomment the line below to start the terminal chat loop:
# interactive_loop()

# ===========================================================
# Section 13 — Optional Gradio Web UI (Bonus)
# ===========================================================
# A minimal Gradio interface — runs in Colab with a public link.
# Uncomment the launch() call to start it.

USE_GRADIO = False   # ← set True to launch Gradio UI

if USE_GRADIO:
    try:
        import gradio as gr

        def gradio_chat(user_message: str, history: list):
            """
            Gradio ChatInterface callback.
            `history` is managed by Gradio; we sync it with our memory buffer.
            """
            # Sync Gradio history → our memory buffer
            conversation_history.clear()
            for user_msg, bot_msg in (history or []):
                conversation_history.append(
                    {"question": user_msg, "answer": bot_msg}
                )
            answer = chat(user_message, verbose=False)
            return answer

        with gr.Blocks(title="Extractive QA Chatbot") as demo:
            gr.Markdown("## 🤖 Extractive QA Chatbot\n"
                        "Ask questions about your uploaded documents.")
            chatbot = gr.ChatInterface(
                fn=gradio_chat,
                examples=[
                    "What is the main mission described in the document?",
                    "When did the event happen?",
                    "Who led that effort?",
                ],
                title="Document QA",
            )

        demo.launch(share=True)   # share=True gives a public Colab link

    except ImportError:
        print("⚠️  Gradio not installed — run: pip install gradio")
    except Exception as e:
        print(f"❌ Gradio launch failed: {e}")

# ===========================================================
# Section 14 — How to Extend
# ===========================================================
"""
How to extend this notebook
────────────────────────────

- Add RAG with a generative model
  Replace the extractive QA pipeline with a seq2seq model such as
  google/flan-t5-base or mistralai/Mistral-7B-Instruct (requires Colab Pro)
  and pass the retrieved context as a prompt prefix.

- Use larger / better models on Colab Pro (A100/V100 GPU)
  Swap QA_MODEL_NAME for deepset/deberta-v3-base-squad2 or
  deepset/roberta-large-squad2 for higher accuracy at the cost of speed.

- Better PDF parsing
  Replace PyPDF2 with unstructured (pip install unstructured[pdf]) to handle
  scanned PDFs, tables, and multi-column layouts more robustly.

- Approximate nearest-neighbour search at scale
  Replace faiss.IndexFlatIP with faiss.IndexIVFFlat or IndexHNSWFlat for
  faster retrieval over millions of chunks.

- Persistent vector database
  Swap the local FAISS file for ChromaDB, Weaviate, or Pinecone so the index
  survives across sessions and can be shared across machines.

- Richer conversational memory
  Implement a sliding-window summariser (e.g., facebook/bart-large-cnn) to
  compress long histories instead of simply truncating to last N turns.

- Streaming answers in Gradio
  Use gr.ChatInterface with streaming=True and a generative model that
  supports token streaming for a more responsive UX.

- Evaluation
  Add a small evaluation loop using the SQuAD metric (pip install evaluate)
  to measure exact-match and F1 on a held-out set of Q/A pairs.

- Metadata filtering
  Store document-level metadata (date, author, category) alongside chunks
  and pre-filter the FAISS search by metadata before semantic ranking.

- Hybrid search
  Combine dense retrieval (FAISS) with sparse BM25 (rank-bm25 library)
  and merge scores with Reciprocal Rank Fusion for better coverage.
"""

print("\n✅ Notebook complete. Scroll up to see demo output.")
print("   Uncomment interactive_loop() (Section 12) to start chatting,")
print("   or set USE_GRADIO = True (Section 13) for the web UI.")


DEMO — 3 Sample QA Interactions

--- Demo 1: Single-chunk factual question ---

🔍 Retrieved 3 chunk(s):
   [2026008103 (1).pdf] score=0.342 — brids). The introduced system includes three major research. The technique initi…
   [2026008103 (1).pdf] score=0.311 — ure. The stages of this modular deep learning architecture include: (1) multi-ca…
   [2026008103 (1).pdf] score=0.296 — ame subject torso/hip bounding size metric, and keypoints (x, y) are normalized …

💬 Q: What is the main mission described in the uploaded document?
✅ A: training hours  (confidence=0.005)

--- Demo 2: Multi-chunk question ---

🔍 Retrieved 3 chunk(s):
   [2026008103 (1).pdf] score=0.387 — size and centered around a BB center. • Augmentation (training hours), which con…
   [2026008103 (1).pdf] score=0.376 — are available, the colors of the uniforms, the colors of the mats, IV. METHODOLO…
   [2026008103 (1).pdf] score=0.360 — brids). The introduced system includes three major research. The technique initi…

💬 Q: